# ML-based downscaling of future climate scenarios

## 1. Setting up

During this workshop we are going to train a deep learning climate downscaling model based on a UNet architecture. We are going to use `EC-Earth3-Veg` global climate model data, interpolated to 1° resolution, as the low resolution input data (**predictors**), and `HCLIM43-ALADIN` regional climate model data, interpolated to 0.1° resolution, as the high resolution output data (**target**).

A graphical processing unit (GPU) is necessary to train such a model within reasonable time. Start the exercises by connecting to a T4 GPU runtime. Click on a dropdown menu next to `Connect` button in the upper right corner, select `Change runtime type`, then `T4 GPU` and press `Save`. After selecting the T4 GPU runtime, proceed to connecting by pressing `Connect T4` button.

You will be given an access to a virtual machine (VM) with a T4 GPU and a small amount of temporary storage. Verify that the connection was successful by executing the next block:

In [ ]:
! nvidia-smi

If the connection has been successful, you should be able to see the output of `nvidia-smi` diagnostics utility showing the `Tesla T4` GPU is available.

## 2. Data download

Download the training data:

In [ ]:
! gdown --folder https://drive.google.com/drive/folders/1Th1kLXkXWZ32M5QFbu1prKSGSF17s0tp -O predictors

In [ ]:
! gdown --folder https://drive.google.com/drive/folders/1uciumbDyqka8LLkkVTaKTz6lnMS0Zjpf -O target

Open the `Files` side panel to access the downloaded files.

You should be able to see `predictors` and `target` folders with multiple `.nc` data files. Let's install a few extra packages and try to inspect a few timesteps from the downloaded climate data:

## 3. Extra package installation

In [ ]:
! pip install netCDF4 scitools-iris palettable

## 4. Climate data plotting

Import a few Python libraries to help with the plotting:

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
import palettable
import cartopy.feature as cfeature

Plot surface temperature `tas` variable from EC-Earth3-Veg model on a single day:

In [ ]:
# Open the dataset
ds = xr.open_dataset('predictors/tas_EC-Earth3-Veg_1990-2050.nc')

In [ ]:
# Select first timestep and convert to Celsius
tas_c = ds["tas"].isel(time=0) - 273.15

In [ ]:
# Create map
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

# Plot
tas_c.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    x="lon",
    y="lat",
    cmap=palettable.cmocean.sequential.Thermal_20.mpl_colormap,
    vmin=WRITE YOUR MIN TEMPERATURE HERE,
    vmax=WRITE YOUR MAX TEMPERATURE HERE,
    cbar_kwargs={"label": "Surface temperature (°C)"}
)

# Add coastlines
ax.coastlines(resolution='50m')

# Set the title
ax.set_title("")

# Show the plot
plt.show()

Now let's compare with the high resolution data from the regional climate model HCLIM:

In [ ]:
# Open the dataset
ds = xr.open_dataset('target/tas_remapped_HCLIM_1990-2050.nc')

In [ ]:
# Select first timestep and convert to Celsius
tas_c = ds["tas"].isel(time=0) - 273.15

In [ ]:
# Create map
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

# Plot
tas_c.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    x="lon",
    y="lat",
    cmap=palettable.cmocean.sequential.Thermal_20.mpl_colormap,
    vmin=WRITE YOUR MIN TEMPERATURE HERE,
    vmax=WRITE YOUR MAX TEMPERATURE HERE,
    cbar_kwargs={"label": "Surface temperature (°C)"}
)

# Add coastlines
ax.coastlines(resolution='50m')

# Set the title
ax.set_title("")

# Show the plot
plt.show()

Essentially, the goal of climate downscaling is to start with low resolution climate data, and increase the resolution to a desired level. While regional climate models run an additional climate model simulation on a finer grid (which requires a lot of computational resources), we are going to train a neural network that will learn a relation between the low and high resolution data, so that it can be used later to produce new high resolution data.

## 5. Standardization of the training data

Now let's download the training scripts and unpack `detex.zip` file:

In [ ]:
! gdown https://drive.google.com/file/d/1a7aqt1sHrwMX_s3cybWNIK8FSGvVPPJ4/view?usp=drive_link --fuzzy -O detex.zip

In [ ]:
! unzip detex.zip -d detex

Before starting the training, we need to standardize the data first to make the training faster and more stable. To do that, we are going to compute monthly means and monthly standard deviations for every variable and every grid point. To standardize the data, we subtract the corresponding mean value and divide by the corresponding standard deviation:

$$
x' = \frac{x - \mu}{\sigma}
$$

where $x'$ is a standardized value $x$, $\mu$ is the monthly mean, and $\sigma$ is the monthly standard deviation.

Thus, the neural network learns to produce standardized high resolution data. To produce the high resolution data in the original space (or units - `K` for surface temperature), we are going to perform the inverse of standardization:

$$
x = \sigma x' + \mu
$$

To perform the standardization, you can use the methods implemented in our `detex` code. Go inside the `detex` directory with the code:

In [ ]:
cd detex

Initially, we are going to train our ML-downscaling model only using the historical data from 1990-2010 period, and test it on the rest of the available data, from 2010 to 2050. The monthly statistics for the standardization are going to be computed in the training period as well. Let's prepare a folder for standardized predictors using the statistics of the 1990-2010 period:

In [ ]:
! mkdir -p ../predictors/1990-2010

Import the libraries for standardization and file handling:

In [ ]:
import os
from detex.standardization import get_local_monthly_stats, standardize_local_monthly, scale_static_field

Standardize the predictors using statistics from 1990-2010 period (training period).

In [ ]:
GCM_name = 'EC-Earth3-Veg'
GCM_period_start = '1990'
GCM_period_end = '2050'
start_date = '1990-01-01'
end_date = '2009-12-31'
rel_path = '../predictors'
std_path = '../predictors/1990-2010'
vars = ['ta', 'ua', 'va', 'hus', 'zg'] # temperature, wind components, specific humidity and geopotential height
levels = [1000, 850, 700, 500] # 1000, 850, 700 and 500 hPa

for var_name in vars:
    for level in levels:
        # Get a dynamic data file name
        ds_name = f"{var_name}_{level}_{GCM_name}_{GCM_period_start}-{GCM_period_end}.nc"
        # Compute the monthly statistics
        get_local_monthly_stats(
            ds_name=ds_name, rel_path=rel_path, std_path=std_path, var_name=var_name, level=level,
            start_date=start_date, end_date=end_date, dataset_name=GCM_name
        )
        # Standardize the data
        stats_file = os.path.join(std_path, f'local_monthly_mean_std_{var_name}_{level}_{GCM_name}_{start_date}_{end_date}.npy')
        standardize_local_monthly(
            ds_name=ds_name, rel_path=rel_path, std_path=std_path, stats_file=stats_file,
            var_name=var_name, x_name='lat', y_name='lon'
        )

Scale the orography to [0, 1] range:

In [ ]:
scale_static_field(
    ds_name='../predictors/orog_remapped_HCLIM.nc',
    out_name='../predictors/scaled.orog_remapped_HCLIM.nc',
    var_name='orog'
)

Move the static data (scaled orography and the land-sea mask with the standardized predictors):

In [ ]:
! cp ../predictors/scaled.orog_remapped_HCLIM.nc ../predictors/1990-2010/
! cp ../predictors/land_sea_mask_remapped_HCLIM.nc ../predictors/1990-2010/

Standardize the target data:

In [ ]:
! mkdir -p ../target/1990-2010

In [ ]:
RCM_name = 'remapped_HCLIM'
RCM_period_start = '1990'
RCM_period_end = '2050'
start_date = '1990-01-01'
end_date = '2009-12-31'
rel_path = '../target'
std_path = '../target/1990-2010'
var = 'tas'

ds_name = f"{var}_{RCM_name}_{RCM_period_start}-{RCM_period_end}.nc"
get_local_monthly_stats(
    ds_name=ds_name, rel_path=rel_path, std_path=std_path, var_name=var,
    start_date=start_date, end_date=end_date, dataset_name=RCM_name
)
stats_file = os.path.join(std_path, f'local_monthly_mean_std_{var}_{RCM_name}_{start_date}_{end_date}.npy')
standardize_local_monthly(
    ds_name=ds_name, rel_path=rel_path, std_path=std_path, stats_file=stats_file,
    var_name=var, x_name='lat', y_name='lon'
)

## 6. Training

Now we are going to configure our first training.

Open the `training-1990-2010-config.toml` configuration file and set the correct paths to the training files. You can edit the configuration file by double-clicking the file in the file manager on your left. A text editor window will appear on your right. Set the correct paths in the `[input]` and `[ground_truth]` sections. Create a suitable `output` directory and configure the `model_path` variable in the `[output]` section of the configuration file.

In [ ]:
! mkdir -p ../output/1990-2010

Check the number of the available CPU cores:

In [ ]:
os.cpu_count()

Start the training in the background:

In [ ]:
! python training.py \
    -c training-1990-2010-config.toml \
    -d cuda \
    -n PUT THE CPU COUNT HERE > train-1990-2010.log 2>&1 &

You can track the progress of the training by 'tailing' the `log` file. The whole training can take about 35 minutes.

In [ ]:
! tail -f train-1990-2010.log

While the training is done using the data from 1990-2010, and the model parameters are updated to minimize the error of the prediction (or the training loss), we run the validation in the 2010-2015 period. That means that we evaluate the model in 2010-2015 period and compute the loss, but do not update the weights. This way, we get an independent measure of the accuracy of our model. Take a look at the `Mean validation loss` for every epoch in the `train-1990-2010.log`. Note how it changes during the training.

Don't forget to stop the tailing process when you are done (it stops only when you stop it).

Save your work to your drive (most importantly the model checkpoints) by mounting your Google Drive to the VM instance. Alternatively, download the files to your computer through the file manager on your left.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Save the `output` directory (containing the model checkpoints) to your Google Drive:

In [ ]:
! mkdir -p /content/drive/MyDrive/ML-climate-downscaling

In [ ]:
! cp -r /content/output/ /content/drive/MyDrive/ML-climate-downscaling/

If you want to restore the model checkpoints (or other data) from your Google Drive, copy the contents to the local VM storage:

In [ ]:
! cp -r /content/drive/MyDrive/ML-climate-downscaling/output /content/

## 7. Inference (predictions)

Run the predictions (inference) using a trained model on 2010-2050 period. The inference will take about 6-7 minutes.

Don't forget to make sure that the correct data paths are set in the `inference-1990-2010-config.toml` file!

In [ ]:
! python inference.py \
    -c inference-1990-2010-config.toml \
    -d cuda \
    -n PUT THE CPU COUNT HERE

It might be worth saving the high resolution predictions in your drive as well:

In [ ]:
! cp /content/output/1990-2010/Predictions_tas_EC-Earth3-Veg_2010-2050.nc /content/drive/MyDrive/ML-climate-downscaling/output/1990-2010/

We can finally analyze the ML-downscaling predictions and compare them to high resolution data from 2010-2050 period. You are free to come up with your own ways to evaluate the performance of the ML-downscaling method, but below you can find a few examples.

## 8. Data analysis

At this point you might consider saving your data in drive, switch to `CPU` instance, load the data from the drive back, and continue analysis to save GPU time. You would need to install the packages again.

### Downscaling example

Let's install `cdo` (Climate Data Operators) tool to quickly manipulate our climate data:

In [ ]:
!apt-get update
!apt-get install -y cdo

We can select just the test period (2010-2050) for easier comparison with the ML predictions:

In [ ]:
! cdo selyear,2010/2050 \
        ../target/tas_remapped_HCLIM_1990-2050.nc \
        ../target/tas_remapped_HCLIM_2010-2050.nc

In [ ]:
! cdo selyear,2010/2050 \
        ../predictors/tas_EC-Earth3-Veg_1990-2050.nc \
        ../predictors/tas_EC-Earth3-Veg_2010-2050.nc

In [ ]:
ds_ml = xr.open_dataset('../output/1990-2010/Predictions_tas_EC-Earth3-Veg_2010-2050.nc')
ds_hclim = xr.open_dataset('../target/tas_remapped_HCLIM_2010-2050.nc')
ds_ece = xr.open_dataset('../predictors/tas_EC-Earth3-Veg_2010-2050.nc')

In [ ]:
# List of datasets and titles
datasets = [ds_ece, ds_hclim, ds_ml]
titles = ["EC-Earth3-Veg", "HCLIM", "ML-downscaling"]

# Create subplots (1 row, 3 columns)
fig, axes = plt.subplots(
    1, 3,
    figsize=(9, 6),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

meshes = []

for ax, ds, title in zip(axes, datasets, titles):
    # Convert temperature to Celsius
    tas_c = ds["tas"].isel(time=0) - 273.15

    mesh = tas_c.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        x="lon",
        y="lat",
        cmap=palettable.cmocean.sequential.Thermal_20.mpl_colormap,
        vmin=WRITE YOUR MIN TEMPERATURE HERE,
        vmax=WRITE YOUR MAX TEMPERATURE HERE,
        add_colorbar=False
    )

    ax.coastlines()
    ax.set_title(title)
    meshes.append(mesh)

# Single horizontal colorbar
cbar = fig.colorbar(
    meshes[0],
    ax=axes,
    orientation="horizontal",
    pad=0.05,
    aspect=45,
    extend='both'
)
cbar.set_label("Surface temperature (°C)")

plt.show()

### Mean annual temperature change

In [ ]:
! cdo -fldmean -yearmean ../predictors/tas_EC-Earth3-Veg_1990-2050.nc ../predictors/yearly_mean_tas_EC-Earth3-Veg_1990-2050.nc

In [ ]:
! cdo -fldmean -yearmean ../target/tas_remapped_HCLIM_1990-2050.nc ../target/yearly_mean_tas_remapped_HCLIM_1990-2050.nc

In [ ]:
! cdo -fldmean -yearmean ../output/1990-2010/Predictions_tas_EC-Earth3-Veg_2010-2050.nc \
../output/1990-2010/yearly_mean_Predictions_tas_EC-Earth3-Veg_2010-2050.nc

In [ ]:
df_hclim = xr.open_dataset('../target/yearly_mean_tas_remapped_HCLIM_1990-2050.nc')
df_ece = xr.open_dataset('../predictors/yearly_mean_tas_EC-Earth3-Veg_1990-2050.nc')
df_ml = xr.open_dataset('../output/1990-2010/yearly_mean_Predictions_tas_EC-Earth3-Veg_2010-2050.nc')

# plot
plt.plot(df_hclim["time"].values, df_hclim["tas"].values.squeeze() - 273.15, label="HCLIM")
plt.plot(df_ece["time"].values, df_ece["tas"].values.squeeze() - 273.15, label="EC-Earth3-Veg")
plt.plot(df_ml["time"].values, df_ml["tas"].values.squeeze() - 273.15, label="ML-downscaling")
plt.axvline(pd.Timestamp("2010-01-01"), color="black", linestyle="--")

plt.xlabel("Year")
plt.ylabel("Mean temperature, °C")
plt.legend()
plt.show()

### Seasonal root-mean squared error

In [ ]:
! cdo sqrt -yseasmean -pow,2 -sub \
        ../output/1990-2010/Predictions_tas_EC-Earth3-Veg_2010-2050.nc \
        ../target/tas_remapped_HCLIM_2010-2050.nc \
        ../output/1990-2010/seasonal_rmse_Predictions_tas_EC-Earth3-Veg_2010-2050.nc

In [ ]:
ds = xr.open_dataset('../output/1990-2010/seasonal_rmse_Predictions_tas_EC-Earth3-Veg_2010-2050.nc')

seasons = ["DJF", "MAM", "JJA", "SON"]

fig, axes = plt.subplots(
    2, 2,
    figsize=(6, 6),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

for i, ax in enumerate(axes.flat):

    ds["tas"].isel(time=i).plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        x="lon",
        y="lat",
        cmap=palettable.cmocean.sequential.Thermal_20.mpl_colormap,
        vmin=WRITE YOUR MIN TEMPERATURE HERE,
        vmax=WRITE YOUR MAX TEMPERATURE HERE,
        add_colorbar=False
    )

    ax.coastlines()
    ax.set_title(seasons[i])

# single colorbar for all plots
cbar = fig.colorbar(
    axes.flat[0].collections[0],
    ax=axes,
    orientation="horizontal",
    pad=0.05,
    extend='max'
)

cbar.set_label("RMSE temperature (°C)")

plt.show()

### Temperature distribution

In [ ]:
! cdo selyear,2030/2050 \
        ../target/tas_remapped_HCLIM_1990-2050.nc \
        ../target/tas_remapped_HCLIM_2030-2050.nc

In [ ]:
! cdo selyear,2030/2050 \
        ../predictors/tas_EC-Earth3-Veg_1990-2050.nc \
        ../predictors/tas_EC-Earth3-Veg_2030-2050.nc

In [ ]:
! cdo selyear,2030/2050 \
        ../output/1990-2010/Predictions_tas_EC-Earth3-Veg_2010-2050.nc \
        ../output/1990-2010/Predictions_tas_EC-Earth3-Veg_2030-2050.nc

In [ ]:
ds_ml = xr.open_dataset('../output/1990-2010/Predictions_tas_EC-Earth3-Veg_2030-2050.nc')
tas_ml = ds_ml["tas"].values.flatten()
tas_ml = tas_ml - 273.15

In [ ]:
ds_hclim_ssp370 = xr.open_dataset('../target/tas_remapped_HCLIM_2030-2050.nc')
tas_hclim_ssp370 = ds_hclim_ssp370["tas"].values.flatten()
tas_hclim_ssp370 = tas_hclim_ssp370 - 273.15

In [ ]:
ds_ece_ssp370 = xr.open_dataset('../predictors/tas_EC-Earth3-Veg_2030-2050.nc')
tas_ece_ssp370 = ds_ece_ssp370["tas"].values.flatten()
tas_ece_ssp370 = tas_ece_ssp370 - 273.15

In [ ]:
cmap = ListedColormap(palettable.colorbrewer.qualitative.Dark2_8.mpl_colors)

plt.figure(figsize=(6, 4))

counts, bin_edges = np.histogram(tas_ml, bins=50, density=True)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
plt.plot(bin_centers, counts, color=cmap(1), lw=2, label='ML-downscaling (2030-2050)')

counts, bin_edges = np.histogram(tas_hclim_ssp370, bins=50, density=True)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
plt.plot(bin_centers, counts, color=cmap(2), lw=2, label='HCLIM (2030-2050)')

counts, bin_edges = np.histogram(tas_ece_ssp370, bins=50, density=True)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
plt.plot(bin_centers, counts, color=cmap(4), lw=2, label='EC-Earth3-Veg (2030-2050)')

plt.xlabel("Temperature (°C)")
plt.ylabel("Probability density")
plt.yscale("log")
plt.ylim(1E-4, 1E-0)
plt.legend()
plt.show()